In [ ]:

import matplotlib.pyplot as plt
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img

# a, b. Load and preprocess data
data = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train = data.flow_from_directory(
    "LP IV Updated Datasets/Object Detection/caltech-101",
    target_size=(224, 224),
    subset="training",
    class_mode="sparse"
)

test = data.flow_from_directory(
    "LP IV Updated Datasets/Object Detection/caltech-101",
    target_size=(224, 224),
    subset="validation",
    class_mode="sparse",
    shuffle=False
)

# c, d. Pre-trained model and freeze layers
base = VGG16(
    weights="LP IV Updated Datasets/Object Detection/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5",
    include_top=False,
    pooling="avg"
)
base.trainable = False

model = Sequential([
    base,
    Dense(train.num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# e. Train model
model.fit(train, epochs=3)

# f, g. Compute loss and evaluate
loss, accuracy = model.evaluate(test)
print("Loss:", loss)
print("Accuracy:", accuracy * 100, "%")

# h. Visualize prediction
prediction = model.predict(test[0][0][:1]).argmax()

plt.imshow(load_img(test.filepaths[0]))
plt.title(list(test.class_indices)[prediction])
plt.show()

# i. Fine-tune model
base.trainable = True

for layer in base.layers[:-4]:
    layer.trainable = False

model.compile(
    optimizer=Adam(0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(train, epochs=1)
